# Model met 6 features

Hier is eerste ontwerp van van het model waarbij we maar 3 features gebruiken, we hebben deze 3 features gekozen:

stm_prioriteit: Dit is de prioriteitsindicatie, dat zegt hoe belangrijk het is dat deze storing snel opgelost moet worden.   
stm_oorz_code: Dit is de oorzaak code, dat geeft aan wat de oorzaak is van de storing.   
stm_sap_meldtijd: Dit is de meldtijd, dat zegt hoelaat de melding is gemaakt.   
stm_aann_t_fh: Dit is onze zelf aangemaakte target variabele, dit is de duur van wanneer de aannemer op locatie is, tot de treinen weer rijden.

stm_contractgeb_gst: Contractgebied van de aannemer   
stm_geo_mld: geocode van de melding    
stm_progfh_in_duur: Duur van hoelang het volgens de aannemer duurt om het probleem op te lossen.   


Om de target variabele te berekenen gebruiken we deze features:   
stm_aanntpl_dd: Datum wanneer aannemer op locatie is.    
stm_aanntpl_tijd: Tijd wanneer aannemer op locatie is.    
stm_fh_dd: Datum wanneer treinen weer rijden.    
stm_fh_tijd: Tijd waneer treinen weer rijden.   

Dit zijn de meetniveaus van de target en features:

stm_prioriteit: ordinaal   
stm_oorz_code: nominaal      
stm_sap_meldtijd: continu  
stm_aann_t_fh: continu    

stm_contractgeb_gst: nominaal   
stm_geo_mld: nominaal   
stm_progfh_in_duur: continu    

stm_aanntpl_dd: continu    
stm_aanntpl_tijd: continu     
stm_fh_dd: continu     
stm_fh_tijd: continu  

Onderzoeksvraag: Kunnen wij met 3 features al een redelijk model maken die kan voorspellen hoelang het duurt voordat de treinen weer rijden. 
Omdat we nu 2 extra nominale waardes hebben, gaan we gebruik maken van Gradient Boosting Regressor. Hiervoor hoeven we niet get_dummies te gebruiken, dus krijgen we uiteindelijk niet 1000 rijen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import pickle
from datetime import datetime
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_csv("sap_storing_data_hu_project.csv", low_memory=False)

## Data verifiëren, omzetten en maken
Eerst kijken we of er duplicates in de dataset zitten

In [ ]:
df["#stm_sap_meldnr"].duplicated().sum()

Er zitten 298799 duplicates in, deze gaan we verwijderen

In [ ]:
df = df.drop_duplicates(subset=["#stm_sap_meldnr"])

### stm_prioriteit
Nu willen we gaan checken of we de features kunnen vertrouwen. Bij de stm_prioriteit kunnen we uit meegegeven documenten bepalen dat deze feature tussen de 1.0 en 9.0 in hele getallen loopt. Om dat te checken gebruiken we deze staafdiagram:

In [ ]:
df['stm_prioriteit'].value_counts().sort_index().plot(kind='bar')

plt.title('Telling van elke unieke feature in stm_prioriteit')
plt.xlabel('stm_prioriteit')
plt.ylabel('Aantal')
plt.grid(axis='y')

Hier kunnen we zien dat wat in het megegeven document staat klopt, en er zitten geen values tussen, dus de data uit deze feature kunnen we vertrouwen. Ook hoeven we deze data niet om te zetten, omdat deze ordinaal is. 

Wat we well moeten doen is alle rijen verwijderen die prioriteit als 9.0 hebben. Deze betekenen dat er geen aannemer naar toe hoeft. Dus is er geen lang genoege storing om dit in de dataset te houden.

In [ ]:
df = df[~df["stm_prioriteit"].isin([9.0])]
df["stm_prioriteit"].describe()

De max is nu 8.0, dit betekent dat alle rijen met prioriteit als 9.0 verwijderd zijn.

### stm_oorz_code
Bij de stm_oorz_code kunnen ook uit een meegegeven document vaststellen wat de verschillende oorzaak codes zijn. Om te kijken of er niet rare values in de kolom zitten, zullen we alle unieke values uit stm_oorz_code	halen, en handmatig checken of die allemaal in het document staan.

In [ ]:
df = df.dropna(subset=["stm_prioriteit", "stm_oorz_code", "stm_sap_meldtijd", "stm_contractgeb_gst", "stm_geo_mld", "stm_progfh_in_duur"])
data = df[["stm_prioriteit", "stm_oorz_code", "stm_sap_meldtijd", "stm_contractgeb_gst", "stm_geo_mld", "stm_progfh_in_duur"]]
print(set(list(data["stm_oorz_code"])))

De volgende values uit die lijst die niet in het document staan zijn: 33, 48, 51 en 999, deze gaan we onderzoeken waarom deze in de dataset staan

In [ ]:
data["stm_oorz_code"].value_counts().loc[[33.0, 48.0, 51.0, 999.0]]

Nu zien we dat 33, 48 en 51 allemaal 1 keer voorkomen, deze zijn dus fouten in de dataset. Deze gaan we verwijderen.
999 komt 102 keer voor, dit is een typische waarde die wordt ingevoerd als de orzaakscode niet duidelijk is. Deze verwijderen we ook.

In [ ]:
data = data[~data["stm_oorz_code"].isin([33.0, 48.0, 51.0, 999.0])]
data["stm_oorz_code"].describe()

We kunnen nu zien dat de min en max van stm_oorz_code valt binnen de values in het meegegeven document.

### stm_sap_meldtijd
Bij stm_sap_meldtijd kunnen we valse tijden hebben, die staan dan als "::" zoals hieronder te zien is. Deze gaan we verwijderen: 

In [ ]:
len(data[data["stm_sap_meldtijd"] == "::"])

In [ ]:
# Verwijderen door alles te verwijderen wat niet lengte 8 is, wat de lengte van de format van de tijd is. (14:00:00)
data = data.dropna(subset=["stm_sap_meldtijd"])
data = data[data["stm_sap_meldtijd"].apply(lambda x: len(x) == 8)]

Omdat de values in deze feature als string staan, moeten we deze omzetten naar een float. Daarom zetten we de stm_sap_meldtijd om naar minuten. Dit zijn dus hoeveel minuten er op dag voorbij zijn gegaan sinds middennacht.

In [ ]:
for index, i in data["stm_sap_meldtijd"].items():
    time_lst = i.split(":") 
    time_in_min = float(time_lst[0]) * 60 + float(time_lst[1]) + float(time_lst[2]) / 60
    data.at[index, "stm_sap_meldtijd"] = time_in_min

In [ ]:
data["stm_sap_meldtijd"] = pd.to_numeric(data["stm_sap_meldtijd"])

data["stm_sap_meldtijd"].describe()

### stm_aann_t_fh
Alle tijden staan nu in minuten, nu gaan we de target aanmaken 

In [ ]:
df1 = df[["stm_aanntpl_dd", "stm_aanntpl_tijd", "stm_fh_dd", "stm_fh_tijd"]].dropna()

# errors='coerce' om alle foute tijden om te zettten naar NaN
df1["stm_aanntpl_dd"] = pd.to_datetime(df1["stm_aanntpl_dd"], format="%d/%m/%Y", errors='coerce')
df1["stm_fh_dd"] = pd.to_datetime(df1["stm_fh_dd"], format="%d/%m/%Y", errors='coerce')

df1["stm_fh_tijd"] = pd.to_datetime(df1["stm_fh_tijd"], format="%H:%M:%S", errors='coerce')
df1["stm_aanntpl_tijd"] = pd.to_datetime(df1["stm_aanntpl_tijd"], format="%H:%M:%S", errors='coerce')

dagen_verschil = ((df1["stm_fh_dd"] - df1["stm_aanntpl_dd"]).dt.days) * 24 * 60
tijden_verschil = (df1["stm_fh_tijd"] - df1["stm_aanntpl_tijd"]).dt.total_seconds() / 60

data["stm_aann_t_fh"] = dagen_verschil + tijden_verschil

# Verwijder alle foute tijden die naar NaN zijn omgezet
data = data.dropna(subset=["stm_aann_t_fh"])

data[["stm_prioriteit", "stm_aann_t_fh", "stm_sap_meldtijd"]].describe()

In [ ]:
data["stm_aann_t_fh"].describe()

Zoals we zien is de max heel hoog en is de min een min getal, uit het interview hebben we de tips gekregen om alles onder de 5 minuten en boven de 8 uur weg te halen.

In [ ]:
data = data[data["stm_aann_t_fh"] <= (8 * 60)] # 8 uur
data = data[data["stm_aann_t_fh"] >= 5] # 5 minuten
data[["stm_prioriteit", "stm_aann_t_fh", "stm_sap_meldtijd"]].describe()

### stm_contractgeb_gst

In [ ]:
print(set(list(data["stm_contractgeb_gst"])))

In [ ]:
data["stm_contractgeb_gst"].value_counts().loc[[17.0, 50.0, 53.0, 54.0, 55.0, 56.0, 57.0, 62.0, 64.0, 82.0, 83.0]]

Ookal staan deze geocodes niet in de dataset, gaan we ze nog steeds houden. Dit doen we omdat van elke geocode die niet in de dataset, er nog steeds een hoop van zijn; er zijn dus geen type fouten

Uit het meegegeven document kunnen we vastellen dat 63, 61, 51, 52, 60, 58, 59 en 70 onbekend zijn. Deze gaan we eerst onderzoeken.

In [ ]:
data["stm_contractgeb_gst"].value_counts().loc[[63, 61, 51, 52, 60, 58, 59, 70]]

Hier zien we dat de onbekende geocodes wel veel waardes hebben, deze hoeven we dus niet te verwijderen.

### stm_geo_mld
Bij stm_geo_mld willen we ook kijken of er verkeerde geocodes in de dataset zitten

In [ ]:
print(sorted(set(data["stm_geo_mld"])))

We kunnen zien dat er ontzetten veel verkeerde waardes in zitten. We zien dat er een float en int versie is van veel geo codes, deze willen we hetzelfde maken

In [ ]:
data['stm_geo_mld'] = data['stm_geo_mld'].astype(float).astype(int)
data['stm_geo_mld'] 

Nu kunnen we zien dat alle values int32 zijn, nu kunnen we gaan kijken of er nog verkeerde geocodes in staan

In [ ]:
print(dict(data["stm_geo_mld"].value_counts()))

Nu zien we dat er van veel geocodes er maar heel weinig van in de database staan. Daarom halen we alle geocodes weg waar er minder dan 5 in de dataset staan.

In [ ]:
data = data[~data["stm_geo_mld"].isin({81, 816, 853, 77, 777, 850, 48, 939, 435, 843, 400, 921, 68, 910, 781, 902, 432, 834, 428, 858})]
data["stm_geo_mld"]

### stm_progfh_in_duur

In [ ]:
data["stm_progfh_in_duur"] = pd.to_numeric(data["stm_progfh_in_duur"], errors="coerce")

data = data[data["stm_progfh_in_duur"] <= (8 * 60)] # 8 uur
data = data[data["stm_progfh_in_duur"] >= 5] # 5 minuten

data[["stm_aann_t_fh", "stm_progfh_in_duur"]].corr()
data

## Model

Als eerst maken we een baseline model met de mediaan en het gemiddelde:

In [ ]:
median = data[["stm_aann_t_fh"]].median()

baseline = [median] * len(data)

print(f"R2 score: {r2_score(data['stm_aann_t_fh'], baseline)}")

RMSE = np.sqrt(mean_squared_error(data['stm_aann_t_fh'], baseline))
print(f"Root Mean Squared Error: {RMSE}")

baseline_info = {'baseline': baseline, 'rmse': RMSE}

with open('base_model.pkl', 'wb') as file:
    pickle.dump(baseline_info, file)

Met een score Baseline van -0.10790995766242273, kunnen we vastellen dat het gemiddelde een betere manier is om de stm_aann_t_fh te voorspellen dan de mediaan. Dit kunnen we zeggen omdat de score Baseline een negatief getal is.

Nu maken we het officiele model

In [ ]:
model = GradientBoostingRegressor()

target = data["stm_aann_t_fh"]
features = data.drop(columns=["stm_aann_t_fh"])

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

X_train['stm_oorz_code'] = pd.Categorical(X_train['stm_oorz_code']).codes
X_test['stm_oorz_code'] = pd.Categorical(X_test['stm_oorz_code']).codes

X_train['stm_contractgeb_gst'] = pd.Categorical(X_train['stm_contractgeb_gst']).codes
X_test['stm_contractgeb_gst'] = pd.Categorical(X_test['stm_contractgeb_gst']).codes

X_train['stm_geo_mld'] = pd.Categorical(X_train['stm_geo_mld']).codes
X_test['stm_geo_mld'] = pd.Categorical(X_test['stm_geo_mld']).codes

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"R2 score: {r2_score(y_test, y_pred)}")

RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean Square Error: {RMSE}")

model_info = {'model': model, 'rmse': RMSE}

with open('gradient_boosting_model.pkl', 'wb') as file:
    pickle.dump(model_info, file)

## Conclusie
Het model met 6 features met een r2 score van 0.5208132146311814 is niet goed genoeg om de target te voorspellen.